# Ablation lab -- which rules carry the edge, which are drag?

**Cutting & isolating** (NotebookLM findings Part 4.1). Every strategy
scores named legs; this lab re-evaluates every leg subset over one
backtest and ranks by out-of-sample profit factor. A subset that beats
the full production set (marked `*`) with a healthy sample means the
missing legs are **drag, not edge** -- deletion candidates.

Two engines agree unless noted:
- `ablate.py` -- event-loop, re-scores leg pass-masks (exact engine logic)
- `ablate_vbt.py` -- vectorbt full-matrix (fast, every rule combination)

Set `STRATEGY` below and run all. Routed strategies (bollinger) **must**
be ablated inside their regime slot -- see the regime cell.

In [ ]:
from pathlib import Path
import subprocess, sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / 'config.json').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from rhengine import viz
viz.use_style()
CFG = json.load(open(ROOT / 'config.json'))
RESULTS = ROOT / 'tests' / 'results'; RESULTS.mkdir(parents=True, exist_ok=True)

# Full-market event-loop passes are slow; cap the universe for a live
# notebook run and raise it (or set 0) when you want the real number.
MAX_TICKERS  = 150
LOOKBACK     = 400
COST_BPS     = 5
REFRESH      = False   # True -> recompute cached CSVs even if present
print('root:', ROOT, '| data_source:', CFG.get('data_source'))

In [ ]:
STRATEGY = 'residualmomentum'   # vwap|bollinger|macd|residualmomentum|pead|consolidation
REGIME   = None                 # e.g. 'choppy' for bollinger's routing slot
SPLIT    = 0.7

## 1 -- Run the event-loop ablation

`ablate.py` runs one backtest at the emission floor (score>=2), records
each trade's leg pass-mask, then qualifies every leg subset. Output is
PF-ranked by OOS.

In [ ]:
cmd = [sys.executable, '-m', 'rhengine.ablate', '--strategy', STRATEGY,
       '--lookback-days', str(LOOKBACK), '--cost-bps', str(COST_BPS),
       '--split', str(SPLIT)]
if REGIME: cmd += ['--regime', REGIME]
print(subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True).stdout)

## 2 -- Chart the leg subsets (in-code re-derivation)

We reproduce the subset table as data so we can chart it. This reuses the
exact `ablate.evaluate_masks` over a fresh capped-universe backtest.

In [ ]:
from rhengine.ablate import evaluate_masks
from rhengine.backtest import run_backtest_on, _iso_date, WARMUP_BARS
from rhengine.indicators import load_spec
from rhengine.data_local import load_bars_any

bars_map, spy_bars = load_bars_any(CFG, lookback_days=LOOKBACK, verbose=False)
if MAX_TICKERS: bars_map = dict(list(bars_map.items())[:MAX_TICKERS])
leg_names = [l['abbr'] for l in load_spec()['strategies'][STRATEGY]['legs']]
dates = [_iso_date(b['time']) for b in spy_bars[WARMUP_BARS:]]
split_date = dates[int(len(dates) * SPLIT)]
trades = run_backtest_on(CFG, bars_map, spy_bars, min_score=2,
                         forced_strategy=STRATEGY, cost_bps=COST_BPS,
                         regime_only=REGIME)
rows = evaluate_masks(trades, len(leg_names), split_date)

def pf(st): return (np.nan if not st else 1e9 if st['profit_factor']==float('inf')
                    else st['profit_factor'])
tbl = pd.DataFrame([{ 'legs': '+'.join(leg_names[i] for i in a),
                      'is_n': (s and s['n']) or 0, 'oos_n': (o and o['n']) or 0,
                      'is_avgR': (s or {}).get('avg_r', np.nan),
                      'oos_avgR': (o or {}).get('avg_r', np.nan),
                      'oos_pf': pf(o), 'oos_t': (o or {}).get('t_stat', np.nan) }
                    for a, s, o in rows])
FULL = '+'.join(leg_names)
tbl = tbl.sort_values('oos_pf', ascending=False).reset_index(drop=True)
tbl

In [ ]:
# OOS avg R per leg subset -- production set outlined. Anything above it
# with a real sample is a cut candidate.
sub = tbl[(tbl.is_n + tbl.oos_n) >= 100].copy()
fig, ax = viz.bars_signed(sub['legs'].tolist(), sub['oos_avgR'].tolist(),
        title=f'{STRATEGY}{" ["+REGIME+"]" if REGIME else ""} -- OOS avg R by leg subset',
        xlabel='OOS avg R (blue>0)', highlight=FULL)
plt.show()

## 3 -- IS vs OOS overfit scan

Subsets in the lower-right wedge win in-sample but lose out-of-sample --
that is curve fit, not edge. The production set should sit on or above
the diagonal.

In [ ]:
s2 = tbl.dropna(subset=['is_avgR', 'oos_avgR'])
fig, ax = viz.is_oos_scatter(s2['is_avgR'], s2['oos_avgR'],
        labels=s2['legs'], title=f'{STRATEGY} leg subsets: IS vs OOS avg R')
plt.show()

## 4 -- Cross-check against the vectorbt full-matrix

`ablation_vbt_results.csv` (committed) is the fast whole-market matrix.
If the two engines disagree on the winning subset, investigate before
trusting either -- this is exactly what happened with bollinger choppy.

In [ ]:
vbt_csv = ROOT / 'ablation_vbt_results.csv'
if vbt_csv.exists():
    vbt = pd.read_csv(vbt_csv)
    v = vbt[vbt.strategy == STRATEGY].sort_values('oos_pf', ascending=False)
    display(v[['rules','n_rules','trades','oos_pf','is_pf','oos_trades']].head(12))
else:
    print('run: py -m rhengine.ablate_vbt  (writes ablation_vbt_results.csv)')

---
**Read-off:** the top OOS-PF subset with >=300 total trades is the lean
configuration. If it drops legs the production set keeps, encode the cut
in `config.json` `legMask` and re-validate in the walk-forward lab before
promoting. Never promote on this table alone -- it is a single split.